# Insect Passage Timeline — Poschiavino River RODI

| Dataset | Station | Position | Period |
|---|---|---|---|
| `250528_post` | RFDO1 | Downstream | Pre-flood (28 May – 2 Jun 2025) |
| `250529_post` | RFUP1 | Upstream   | Pre-flood (28 May – 3 Jun 2025) |
| `250609_post` | RFDO1 | Downstream | Post-flood (9–15 Jun 2025) |
| `250609_RFUP1`| RFUP1 | Upstream   | Post-flood (9–15 Jun 2025) |

**Method:**
1. Insect ROI timestamps are read directly from the RODI metadata CSVs.
2. ROIs are binned into **30-minute** blocks.
3. The FP rate measured on 400 manually reviewed samples per 6 h block is assumed uniform within each 6 h window and applied to all its 30-min sub-blocks.
4. Net-cleaning windows are shaded **grey** (documented, ~10–20 min).
5. Extended no-net windows (repair/deployment) are shaded **orange**.
6. Unexplained zero-count windows within a deployment period are shaded **yellow**.
7. RFUP1 datasets have no net-cleaning CSV yet — their entire background is **light yellow** as a reminder.

## 1 · Imports & Configuration

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'font.size':         12,
    'axes.titlesize':    14,
    'axes.labelsize':    12,
    'xtick.labelsize':   11,
    'ytick.labelsize':   11,
    'legend.fontsize':   11,
    'lines.linewidth':   2.0,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.facecolor':  'white',
    'axes.facecolor':    'white',
    'savefig.facecolor': 'white',
    'savefig.bbox':      'tight',
})

BASE        = Path('data')
FP_BASE     = BASE / 'fp_rates'
CLEANING_CSV = BASE / 'Group_2_Net_Cleaning.csv'
OUT_DIR     = Path('figures')
OUT_DIR.mkdir(exist_ok=True)

BLOCK_MIN = 30   # bin width in minutes

# ── dataset metadata ──────────────────────────────────────────────────────────
DATASETS = {
    '250528_post': {
        'station': 'RFDO1', 'position': 'Downstream', 'period': 'Pre-flood',
        'fish_csv':   None,  # not needed when timeline cache exists
        'roi_root':   None,  # not needed when timeline cache exists
        'fp_csv':     FP_BASE / '250528_post' / '250528_post_sampling_summary.csv',
        'fp_sep':     ';',
        'has_cleaning': True,
        'cleaning_loc': 'RFDO_1',
    },
    '250529_post': {
        'station': 'RFUP1', 'position': 'Upstream', 'period': 'Pre-flood',
        'fish_csv':   None,  # not needed when timeline cache exists
        'roi_root':   None,  # not needed when timeline cache exists
        'fp_csv':     FP_BASE / '250529_post' / 'sampling_summary_post.csv',
        'fp_sep':     ';',
        'has_cleaning': False,
        'cleaning_loc': None,
    },
    '250609_post': {
        'station': 'RFDO1', 'position': 'Downstream', 'period': 'Post-flood',
        'fish_csv':   None,  # not needed when timeline cache exists
        'roi_root':   None,  # not needed when timeline cache exists
        'fp_csv':     FP_BASE / '250609_post' / '250609_post_sampling_summary.csv',
        'fp_sep':     ',',
        'has_cleaning': True,
        'cleaning_loc': 'RFDO_1',
    },
    '250609_RFUP1': {
        'station': 'RFUP1', 'position': 'Upstream', 'period': 'Post-flood',
        'fish_csv':   None,  # not needed when timeline cache exists
        'roi_root':   None,  # not needed when timeline cache exists
        'fp_csv':     FP_BASE / '250609_RFUP1' / 'sampling_summary.csv',
        'fp_sep':     ',',
        'has_cleaning': False,
        'cleaning_loc': None,
    },
}

COLORS = {
    '250528_post':  '#1f77b4',
    '250529_post':  '#2ca02c',
    '250609_post':  '#ff7f0e',
    '250609_RFUP1': '#d62728',
}
LABELS = {
    '250528_post':  'RFDO1 Pre-flood (DS)',
    '250529_post':  'RFUP1 Pre-flood (US)',
    '250609_post':  'RFDO1 Post-flood (DS)',
    '250609_RFUP1': 'RFUP1 Post-flood (US)',
}

## 2 · Build ROI Timestamp Index

In [13]:
def build_image_index(roi_root: Path) -> pd.DataFrame:
    """Collect image_name → local_timestamp from all metadata CSVs under roi_root."""
    records = []
    for meta_csv in roi_root.glob('*/ROIs_*/metadata_*.csv'):
        try:
            header = pd.read_csv(meta_csv, nrows=0).columns.tolist()
            ts_col = 'local_timestamp' if 'local_timestamp' in header else 'utc_timestamp'
            meta = pd.read_csv(meta_csv, usecols=['image_name', ts_col])
            meta = meta.rename(columns={ts_col: 'local_timestamp'})
            ts = pd.to_datetime(meta['local_timestamp'], format='mixed')
            if ts.dt.tz is not None:
                ts = ts.dt.tz_localize(None)
            meta['local_timestamp'] = ts
            records.append(meta)
        except Exception as e:
            print(f'  [warn] {meta_csv.name}: {e}')
    if not records:
        print(f'  [warn] No metadata found in {roi_root}')
        return pd.DataFrame(columns=['image_name', 'local_timestamp'])
    df = pd.concat(records, ignore_index=True)
    df = df.drop_duplicates('image_name')
    return df


CACHE_DIR = BASE / 'notebook_cache'
CACHE_DIR.mkdir(exist_ok=True)

# Fast path: if all annotated timelines are already cached, skip everything below
_timeline_files = {tag: CACHE_DIR / f'timeline_{tag}.csv' for tag in DATASETS}
_cache_hit = all(f.exists() for f in _timeline_files.values())

if _cache_hit:
    print('All timeline caches found — loading directly, skipping index build.')
    TS_INDEX = {}
    TIMELINES = {}
    for tag in DATASETS:
        TIMELINES[tag] = pd.read_csv(
            _timeline_files[tag],
            parse_dates=['bin_start', 'bin_end'],
        )
        print(f'  {tag}: {len(TIMELINES[tag])} bins loaded from cache')
else:
    print('Building timestamp indices (may take a moment for large datasets)...')
    TS_INDEX = {}
    for tag, meta in DATASETS.items():
        _idx_file = CACHE_DIR / f'ts_index_{tag}.csv'
        if _idx_file.exists():
            print(f'  {tag}... loaded from cache', end=' ')
            TS_INDEX[tag] = pd.read_csv(_idx_file, parse_dates=['local_timestamp'])
        else:
            print(f'  {tag}...', end=' ', flush=True)
            TS_INDEX[tag] = build_image_index(meta['roi_root'])
            TS_INDEX[tag].to_csv(_idx_file, index=False)
        print(f'{len(TS_INDEX[tag]):,} ROIs indexed')

All timeline caches found — loading directly, skipping index build.
  250528_post: 237 bins loaded from cache
  250529_post: 274 bins loaded from cache
  250609_post: 289 bins loaded from cache
  250609_RFUP1: 290 bins loaded from cache


## 3 · Load Insect ROIs and Join Timestamps

In [14]:
def load_insects_with_timestamps(tag: str) -> pd.DataFrame:
    meta  = DATASETS[tag]
    df    = pd.read_csv(meta['fish_csv'])
    insects = df[df['predicted_class'] == 'insect'].copy()
    insects['image_name'] = insects['image_path'].apply(lambda p: Path(p).name)

    merged = insects.merge(TS_INDEX[tag], on='image_name', how='left')
    merged = merged.dropna(subset=['local_timestamp'])
    merged['tag'] = tag
    print(f'  {tag}: {len(df):,} total ROIs, {len(insects):,} insects, '
          f'{len(merged):,} with timestamps')
    return merged[['tag', 'image_name', 'local_timestamp']]


if not _cache_hit:
    print('Loading insect ROIs...')
    RAW = {tag: load_insects_with_timestamps(tag) for tag in DATASETS}

## 4 · Load 6 h FP Rates

In [15]:
def parse_block_ts(block_str: str) -> pd.Timestamp:
    """'2025-05-28_18h-24h' → Timestamp at block start."""
    date_part, time_part = block_str.split('_')
    start_h = int(time_part.split('h')[0])
    return pd.Timestamp(f'{date_part} {start_h:02d}:00')


FP_RATES = {}
for tag, meta in DATASETS.items():
    df = pd.read_csv(meta['fp_csv'], sep=meta['fp_sep'])
    df['block_start'] = df['block'].apply(parse_block_ts)
    df['block_end']   = df['block_start'] + pd.Timedelta(hours=6)
    df['fp_rate']     = df['false_positive'] / df['sampled']
    df['tp_rate']     = 1 - df['fp_rate']
    FP_RATES[tag] = df[['block_start', 'block_end', 'fp_rate', 'tp_rate',
                         'total_insects', 'sampled', 'false_positive']].copy()

print('FP rate tables loaded:')
for tag, df in FP_RATES.items():
    print(f'  {tag}: {len(df)} blocks, FP rate range '
          f'{df["fp_rate"].min()*100:.0f}%–{df["fp_rate"].max()*100:.0f}%')

FP rate tables loaded:
  250528_post: 20 blocks, FP rate range 29%–65%
  250529_post: 24 blocks, FP rate range 22%–100%
  250609_post: 25 blocks, FP rate range 0%–100%
  250609_RFUP1: 26 blocks, FP rate range 34%–100%


## 5 · Bin ROIs into 30-min Blocks and Apply FP Correction

In [16]:
def make_30min_timeline(tag: str) -> pd.DataFrame:
    insects = RAW[tag]
    if insects.empty:
        return pd.DataFrame()

    freq = f'{BLOCK_MIN}min'
    insects = insects.set_index('local_timestamp').sort_index()
    counts = insects.resample(freq).size().rename('raw_count').reset_index()
    counts = counts.rename(columns={'local_timestamp': 'bin_start'})
    counts['bin_end'] = counts['bin_start'] + pd.Timedelta(minutes=BLOCK_MIN)

    fp = FP_RATES[tag].copy()

    def get_fp_rate(ts):
        mask = (fp['block_start'] <= ts) & (ts < fp['block_end'])
        row  = fp[mask]
        return row['tp_rate'].iloc[0] if len(row) else np.nan

    counts['tp_rate']   = counts['bin_start'].apply(get_fp_rate)
    counts['fp_rate']   = 1 - counts['tp_rate']
    counts['est_true']  = (counts['raw_count'] * counts['tp_rate']).fillna(0).round().astype(int)
    counts['tag']       = tag
    counts['station']   = DATASETS[tag]['station']
    counts['position']  = DATASETS[tag]['position']
    counts['period']    = DATASETS[tag]['period']
    return counts


if not _cache_hit:
    TIMELINES = {tag: make_30min_timeline(tag) for tag in DATASETS}

    for tag, tl in TIMELINES.items():
        if not tl.empty:
            print(f'{tag}: {len(tl)} bins, '
                  f'{tl["bin_start"].min().date()} – {tl["bin_start"].max().date()}, '
                  f'peak est. true insects/30min = {tl["est_true"].max():,}')

## 6 · Parse Net-Cleaning Intervals

In [ ]:
def safe_time(date: pd.Timestamp, time_str, prev_dt=None) -> pd.Timestamp | None:
    """Combine date + 'HH:MM' string; if result < prev_dt advance date by 1 day."""
    if pd.isna(time_str) or str(time_str).strip() == '':
        return None
    t = pd.Timedelta(str(time_str).strip() + ':00')
    dt = date + t
    if prev_dt is not None and dt < prev_dt:
        dt += pd.Timedelta(days=1)
    return dt


def parse_cleaning_csv(csv_path: Path, year: int = 2025):
    """
    Returns:
      cleaning_intervals : list of (start, end, location_id)
      no_net_intervals   : list of (start, end, location_id)
      data_start_no_net  : list of (end, location_id)  – net only available from 'end'
    """
    df = pd.read_csv(csv_path, sep=';')
    df.columns = [c.strip() for c in df.columns]
    df = df.rename(columns={
        'Location ID':          'loc',
        'Time before cleaning': 'before',
        'Time after cleaning':  'after',
    })
    df['date'] = pd.to_datetime(
        df['Day'].str.strip() + f'-{year}', format='%d-%b-%Y', errors='coerce'
    )
    df = df[df['date'].notna()].copy()

    cleaning_intervals  = []
    no_net_intervals    = []
    data_start_no_net   = []
    pending_removal     = {}

    for _, row in df.iterrows():
        loc  = str(row['loc']).strip()
        date = row['date']
        t_before = safe_time(date, row.get('before'))
        t_after  = safe_time(date, row.get('after'), prev_dt=t_before)

        if t_before is not None and t_after is not None:
            cleaning_intervals.append((t_before, t_after + pd.Timedelta(minutes=5), loc))
        elif t_before is not None and t_after is None:
            pending_removal[loc] = t_before
        elif t_before is None and t_after is not None:
            end_dt = t_after + pd.Timedelta(minutes=5)
            if loc in pending_removal:
                no_net_intervals.append((pending_removal.pop(loc), end_dt, loc))
            else:
                data_start_no_net.append((end_dt, loc))

    return cleaning_intervals, no_net_intervals, data_start_no_net


CLEANING, NO_NET, DATA_START_NO_NET = parse_cleaning_csv(CLEANING_CSV)

# ── Hardcoded flood gap ───────────────────────────────────────────────────────
# Net assumed absent after last pre-flood cleaning (01-Jun 12:14) until
# reinstallation on 12-Jun 11:50. No removal was logged; this is an assumption.
NO_NET.append((
    pd.Timestamp('2025-06-01 12:14'),
    pd.Timestamp('2025-06-12 11:50'),
    'RFDO_1',
))


print(f'{len(CLEANING)} routine cleaning intervals')
print(f'{len(NO_NET)} extended no-net intervals:')
for s, e, loc in NO_NET:
    print(f'  {loc}  {s}  →  {e}')
print(f'{len(DATA_START_NO_NET)} deployment-start no-net cases:')
for e, loc in DATA_START_NO_NET:
    print(f'  {loc}: net available from {e}')

## 7 · Annotate Each 30-min Bin

In [ ]:
def overlaps_any(ts_start, ts_end, intervals):
    """True if [ts_start, ts_end] overlaps any (start, end, *_) interval."""
    for iv in intervals:
        iv_start, iv_end = iv[0], iv[1]
        if ts_start < iv_end and ts_end > iv_start:
            return True
    return False


def annotate_timeline(tag: str, tl: pd.DataFrame) -> pd.DataFrame:
    tl = tl.copy()
    meta = DATASETS[tag]

    if not meta['has_cleaning']:
        tl['status'] = 'unknown'
        return tl

    loc = meta['cleaning_loc']
    cleaning = [(s, e) for s, e, l in CLEANING   if l == loc]
    no_net   = [(s, e) for s, e, l in NO_NET     if l == loc]

    deploy_start_no_net = [
        e for e, l in DATA_START_NO_NET
        if l == loc
        and tl['bin_start'].min() <= e <= tl['bin_start'].max()
    ]

    def classify(row):
        bs, be = row['bin_start'], row['bin_end']
        if overlaps_any(bs, be, [(s, e) for s, e in no_net]):
            return 'no_net'
        if overlaps_any(bs, be, [(s, e) for s, e in cleaning]):
            return 'cleaning'
        if deploy_start_no_net and bs < min(deploy_start_no_net):
            return 'no_net'
        return 'normal'

    tl['status'] = tl.apply(classify, axis=1)
    return tl


if not _cache_hit:
    for tag in DATASETS:
        TIMELINES[tag] = annotate_timeline(tag, TIMELINES[tag])

    # Save annotated timelines so the next restart skips all of the above
    for tag, tl in TIMELINES.items():
        tl.to_csv(_timeline_files[tag], index=False)
    print(f'Annotated timelines saved to cache ({CACHE_DIR})')

for tag, tl in TIMELINES.items():
    counts = tl['status'].value_counts().to_dict()
    print(f'{tag}: {counts}')

## 8 · Timeline Plots

One subplot per dataset, with shading for each operational status.

In [ ]:
import matplotlib.ticker as mticker

STATUS_COLORS = {
    'cleaning': ('#bbbbbb', 'lightgrey',    'Cleaning'),
    'no_net':   ('#ff8c00', 'moccasin',      'No net / repair'),
    'unknown':  ('#e8c200', 'lightyellow',   'No cleaning data'),
    'normal':   (None,       None,           'Normal operation'),
}


def merge_intervals(intervals):
    """Merge overlapping (start, end) intervals so no region is drawn twice."""
    if not intervals:
        return []
    sorted_ivs = sorted(intervals, key=lambda x: x[0])
    merged = list(sorted_ivs[:1])
    for start, end in sorted_ivs[1:]:
        if start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged


def shade_intervals(ax, intervals, facecolor, alpha=0.4, hatch=None, zorder=0):
    """Fill vertical bands on ax for each (start, end) interval."""
    for start, end in intervals:
        ax.axvspan(start, end, facecolor=facecolor, alpha=alpha,
                   hatch=hatch, edgecolor='none', zorder=zorder)


def _apply_date_xaxis(ax, show_labels=True):
    """Major ticks at midnight (DD Mon, rotated 45°), minor ticks at 06/12/18h."""
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
    ax.xaxis.set_minor_locator(mdates.HourLocator(byhour=[6, 12, 18]))
    ax.xaxis.set_minor_formatter(mticker.FuncFormatter(
        lambda x, _: f"{mdates.num2date(x).replace(tzinfo=None).hour:02d}h"
    ))
    plt.setp(ax.xaxis.get_majorticklabels(), fontsize=7, rotation=45, ha='right',
             visible=show_labels)
    plt.setp(ax.xaxis.get_minorticklabels(), fontsize=6, visible=show_labels)


def plot_dataset_timeline(ax, tag, show_fp_rate=True, show_xlabels=True):
    tl   = TIMELINES[tag]
    meta = DATASETS[tag]
    color = COLORS[tag]
    loc   = meta.get('cleaning_loc')

    # ── background shading ───────────────────────────────────────────────────
    if not meta['has_cleaning']:
        ax.set_facecolor('#fffde7')
    else:
        nn_ivs = [(s, e) for s, e, l in NO_NET if l == loc]
        if not tl.empty:
            data_start = tl['bin_start'].min()
            data_end   = tl['bin_start'].max() + pd.Timedelta(minutes=BLOCK_MIN)
            for end_dt, l in DATA_START_NO_NET:
                if l == loc and data_start < end_dt <= data_end:
                    nn_ivs.append((data_start, end_dt))
        shade_intervals(ax, merge_intervals(nn_ivs), '#ff8c00', alpha=0.35, zorder=1)

        c_ivs = [(s, e) for s, e, l in CLEANING if l == loc]
        shade_intervals(ax, c_ivs, '#cccccc', alpha=0.6, zorder=2)

    # ── bars: normal bins ────────────────────────────────────────────────────
    normal = tl[tl['status'] == 'normal']
    ax.bar(normal['bin_start'], normal['est_true'],
           width=pd.Timedelta(minutes=BLOCK_MIN - 2),
           color=color, alpha=0.75, align='edge', zorder=3)

    # ── bars: unknown status (RFUP1) ─────────────────────────────────────────
    unknown = tl[tl['status'] == 'unknown']
    ax.bar(unknown['bin_start'], unknown['est_true'],
           width=pd.Timedelta(minutes=BLOCK_MIN - 2),
           color=color, alpha=0.5, align='edge', zorder=3)

    # ── bars: no-net bins (included, hatched to flag reduced data quality) ───
    no_net_bins = tl[tl['status'] == 'no_net']
    ax.bar(no_net_bins['bin_start'], no_net_bins['est_true'],
           width=pd.Timedelta(minutes=BLOCK_MIN - 2),
           color=color, alpha=0.4, align='edge', zorder=3, hatch='//')

    # ── secondary axis: FP rate ───────────────────────────────────────────────
    if show_fp_rate:
        ax2 = ax.twinx()
        ax2.plot(tl['bin_start'], tl['fp_rate'] * 100,
                 color='#888888', lw=0.8, ls='--', alpha=0.7, zorder=4)
        ax2.set_ylim(0, 110)
        ax2.set_ylabel('FP rate (%)', fontsize=7, color='#888888')
        ax2.tick_params(axis='y', labelsize=7, labelcolor='#888888')
        ax2.spines['top'].set_visible(False)

    ax.set_title(f'{LABELS[tag]}', fontsize=10, fontweight='bold')
    ax.set_ylabel('Est. true insects / 30 min')
    _apply_date_xaxis(ax, show_labels=show_xlabels)


fig, axes = plt.subplots(4, 1, figsize=(16, 14), constrained_layout=True,
                         sharey=True, sharex=True)

for i, (ax, tag) in enumerate(zip(axes, DATASETS)):
    plot_dataset_timeline(ax, tag, show_xlabels=(i == len(axes) - 1))

# shared legend
legend_patches = [
    mpatches.Patch(facecolor='#cccccc', label='Net cleaning (~10–20 min)'),
    mpatches.Patch(facecolor='#ff8c00', alpha=0.5, label='No net / repair / pre-deployment'),
    mpatches.Patch(facecolor='#fffde7', label='No cleaning data (RFUP1)'),
    mpatches.Patch(color='#888888', label='FP rate % (dashed, right axis)'),
    mpatches.Patch(facecolor='#888888', alpha=0.4, hatch='//', label='No-net bins (included, hatched)'),
]
axes[0].legend(handles=legend_patches, loc='upper right', fontsize=8, framealpha=0.85)

fig.suptitle('Insect passage timeline — 30-min bins, FP-corrected', fontsize=12)
fig.savefig(OUT_DIR / 'insect_timeline_30min.png', bbox_inches='tight')
plt.show()

## 9 · Upstream vs Downstream — Overlaid (same period)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 8), constrained_layout=True)

for ax, (period, ds_tag, us_tag) in zip(
    axes,
    [('Pre-flood',  '250528_post',  '250529_post'),
     ('Post-flood', '250609_post',  '250609_RFUP1')]
):
    ax2 = ax.twinx()
    for this_ax, tag, alpha in [(ax, ds_tag, 0.7), (ax2, us_tag, 0.7)]:
        tl     = TIMELINES[tag]
        normal = tl[tl['status'].isin(['normal', 'unknown', 'no_net'])]
        this_ax.bar(
            normal['bin_start'], normal['est_true'],
            width=pd.Timedelta(minutes=BLOCK_MIN - 2),
            color=COLORS[tag], alpha=alpha, align='edge', label=LABELS[tag]
        )
        this_ax.set_ylabel(f'{LABELS[tag]}', fontsize=8, color=COLORS[tag])
        this_ax.tick_params(axis='y', labelcolor=COLORS[tag])
        this_ax.spines['top'].set_visible(False)

    # Collect all no-net intervals (regular + pre-deployment), merge, then draw once.
    loc = DATASETS[ds_tag]['cleaning_loc']
    nn_ivs = [(s, e) for s, e, l in NO_NET if l == loc]
    tl_ds = TIMELINES[ds_tag]
    if not tl_ds.empty:
        ds_start = tl_ds['bin_start'].min()
        ds_end   = tl_ds['bin_start'].max() + pd.Timedelta(minutes=BLOCK_MIN)
        for end_dt, l in DATA_START_NO_NET:
            if l == loc and ds_start < end_dt <= ds_end:
                nn_ivs.append((ds_start, end_dt))
    shade_intervals(ax, merge_intervals(nn_ivs), '#ff8c00', 0.3, zorder=1)
    shade_intervals(ax, [(s, e) for s, e, l in CLEANING if l == loc], '#cccccc', 0.5, zorder=2)
    ax2.set_facecolor('#fffde7')  # yellow tint for RFUP1

    ax.set_title(f'{period} — Downstream (left axis) vs Upstream (right axis)',
                 fontsize=10, fontweight='bold')
    _apply_date_xaxis(ax)

    # combined legend
    lines = ([mpatches.Patch(color=COLORS[ds_tag], label=LABELS[ds_tag]),
               mpatches.Patch(color=COLORS[us_tag], label=LABELS[us_tag])])
    ax.legend(handles=lines, fontsize=8, loc='upper right')

fig.suptitle('Upstream vs Downstream insect passage — independent y-axes', fontsize=12)
fig.savefig(OUT_DIR / 'up_vs_downstream_30min.png', bbox_inches='tight')
plt.show()

## 10 · Diurnal Pattern (30-min resolution, averaged across days)

In [ ]:
# 'normal', 'unknown', and 'no_net' bins all contribute; only 'cleaning' excluded
all_normal = pd.concat(
    [tl[tl['status'].isin(['normal', 'unknown', 'no_net'])].assign(tag=tag)
     for tag, tl in TIMELINES.items()],
    ignore_index=True
)
all_normal['time_of_day'] = (
    all_normal['bin_start'].dt.hour + all_normal['bin_start'].dt.minute / 60
)

fig, ax = plt.subplots(figsize=(12, 4))
for tag, grp in all_normal.groupby('tag'):
    diurnal = grp.groupby('time_of_day')['est_true'].agg(['mean', 'sem']).reset_index()
    ax.plot(diurnal['time_of_day'], diurnal['mean'],
            lw=1.8, color=COLORS[tag], label=LABELS[tag])
    ax.fill_between(
        diurnal['time_of_day'],
        diurnal['mean'] - diurnal['sem'],
        diurnal['mean'] + diurnal['sem'],
        color=COLORS[tag], alpha=0.2
    )

ax.set_xticks(range(0, 25, 2))
ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 2)])
ax.set_xlabel('Time of day')
ax.set_ylabel('Mean est. true insects / 30 min (± SEM across days)')
ax.set_title('Diurnal insect passage pattern — normal & no-net bins (cleaning excluded)')
ax.legend(fontsize=9)

fig.tight_layout()
fig.savefig(OUT_DIR / 'diurnal_pattern_30min.png', bbox_inches='tight')
plt.show()

## 11 · Summary Table

In [ ]:
rows = []
for tag, tl in TIMELINES.items():
    active = tl[tl['status'].isin(['normal', 'unknown', 'no_net'])]
    fp_df  = FP_RATES[tag]
    peak_bin = active.loc[active['est_true'].idxmax()] if not active.empty else None
    rows.append({
        'Dataset':            tag,
        'Station':            DATASETS[tag]['station'],
        'Period':             DATASETS[tag]['period'],
        'Active 30-min bins': len(active),
        'Cleaning bins':      (tl['status'] == 'cleaning').sum(),
        'No-net bins':        (tl['status'] == 'no_net').sum(),
        'Total est. insects': int(active['est_true'].sum()),
        'Peak insects/30min': int(active['est_true'].max()) if not active.empty else 0,
        'Peak bin':           str(peak_bin['bin_start']) if peak_bin is not None else '',
        'Mean FP rate (%)':   round(fp_df['fp_rate'].mean() * 100, 1),
    })

summary = pd.DataFrame(rows)
display(summary)

## 12 · Day × Hour Activity Heatmap

One subplot per dataset. Rows = calendar days, columns = 30-min bins.  
Grey cells = net-cleaning (excluded). Orange tint = no-net / repair period.

In [ ]:
import matplotlib.colors as mcolors

_base_cmap = plt.cm.YlOrRd.copy()
_base_cmap.set_bad('#cccccc')          # grey for cleaning bins
_nn_cmap   = mcolors.ListedColormap(['#ff8c00'])  # orange overlay for no-net

full_hours = np.arange(0, 24, 0.5)    # 48 half-hour slots

fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)

for ax, tag in zip(axes.flat, DATASETS):
    tl = TIMELINES[tag].copy()
    tl['date']  = tl['bin_start'].dt.date
    tl['tod_h'] = tl['bin_start'].dt.hour + tl['bin_start'].dt.minute / 60

    # cleaning → NaN (grey); all other statuses keep their corrected count
    tl['val'] = tl['est_true'].astype(float)
    tl.loc[tl['status'] == 'cleaning', 'val'] = np.nan

    # pivot: rows = date, cols = 30-min time-of-day slot
    pivot = (
        tl.pivot_table(index='date', columns='tod_h', values='val', aggfunc='sum')
          .reindex(columns=full_hours)
    )

    # no-net mask (1.0 where no_net, NaN elsewhere) for orange overlay
    _st_pivot = (
        tl.pivot_table(index='date', columns='tod_h', values='status', aggfunc='first')
          .reindex(columns=full_hours)
    )
    nonet_grid = pd.DataFrame(
        np.where(_st_pivot.values == 'no_net', 1.0, np.nan),
        index=_st_pivot.index, columns=_st_pivot.columns,
    )

    dates = list(pivot.index)

    im = ax.imshow(pivot.values, aspect='auto', cmap=_base_cmap,
                   interpolation='none', origin='upper', vmin=0)
    ax.imshow(nonet_grid.values, aspect='auto', cmap=_nn_cmap,
              interpolation='none', origin='upper', alpha=0.30,
              vmin=0.5, vmax=1.5)

    # x-axis: label every 2 hours
    tick_idx = [i for i, h in enumerate(full_hours) if h % 2 == 0]
    ax.set_xticks(tick_idx)
    ax.set_xticklabels(
        [f'{int(h):02d}:00' for h in full_hours if h % 2 == 0],
        fontsize=7, rotation=45, ha='right',
    )

    # y-axis: one row per calendar day
    ax.set_yticks(range(len(dates)))
    ax.set_yticklabels([str(d) for d in dates], fontsize=8)

    ax.set_title(LABELS[tag], fontsize=10, fontweight='bold')
    ax.set_xlabel('Time of day', fontsize=8)

    cb = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
    cb.set_label('Est. insects / 30 min', fontsize=7)
    cb.ax.tick_params(labelsize=7)

legend_handles = [
    mpatches.Patch(facecolor='#cccccc', label='Net cleaning (excluded)'),
    mpatches.Patch(facecolor='#ff8c00', alpha=0.5, label='No net / repair'),
]
axes[0, 0].legend(handles=legend_handles, loc='upper right', fontsize=8, framealpha=0.85)

fig.suptitle('Insect activity heatmap — day × 30-min bin, FP-corrected', fontsize=12)
fig.savefig(OUT_DIR / 'heatmap_day_hour.png', bbox_inches='tight')
plt.show()

## 13 · Pre-flood Station Comparison  
Both stations during the overlapping pre-flood window. Pearson r measures synchrony (shared emergence signal vs. independent behaviour).

In [ ]:
# Pre-flood overlapping window — time series + synchrony scatter
pre_ds = TIMELINES['250528_post'].copy()   # RFDO1
pre_us = TIMELINES['250529_post'].copy()   # RFUP1

ds_act = pre_ds[pre_ds['status'].isin(['normal', 'no_net'])].set_index('bin_start')
us_act = pre_us[pre_us['status'].isin(['normal', 'unknown'])].set_index('bin_start')

ov_start = max(ds_act.index.min(), us_act.index.min())
ov_end   = min(ds_act.index.max(), us_act.index.max())

ds_ov = ds_act.loc[ov_start:ov_end, 'est_true']
us_ov = us_act.loc[ov_start:ov_end, 'est_true']

paired_pre = pd.DataFrame({'DS': ds_ov, 'US': us_ov}).dropna()
r_pre = np.corrcoef(paired_pre['DS'], paired_pre['US'])[0, 1]
n_pre = len(paired_pre)
t_stat = r_pre * np.sqrt(n_pre - 2) / np.sqrt(max(1 - r_pre**2, 1e-12))
from scipy.stats import t as _t, mannwhitneyu
p_pre = 2 * _t.sf(abs(t_stat), df=n_pre - 2)
p_str = 'p < 0.001' if p_pre < 0.001 else f'p = {p_pre:.3f}'

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

# Left: time series overlay
ax = axes[0]
ax.plot(ds_ov.index, ds_ov.values, color=COLORS['250528_post'], lw=2,
        label='RFDO1  Downstream')
ax.plot(us_ov.index, us_ov.values, color=COLORS['250529_post'], lw=2,
        label='RFUP1  Upstream', alpha=0.85)
_apply_date_xaxis(ax)
ax.set_ylabel('Est. insects / 30 min')
ax.set_title(f'Pre-flood — both stations\n'
             f'{ov_start.strftime("%d %b")} – {ov_end.strftime("%d %b %Y")}',
             fontweight='bold')
ax.legend()

# Right: scatter + 1:1 line
ax = axes[1]
ax.scatter(paired_pre['DS'], paired_pre['US'],
           alpha=0.45, color='steelblue', s=30, edgecolors='none')
lim = max(paired_pre.max().max() * 1.1, 1)
ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.4, label='1:1 line')
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('RFDO1  Downstream (est. insects / 30 min)')
ax.set_ylabel('RFUP1  Upstream (est. insects / 30 min)')
ax.set_title(f'Pre-flood synchrony\nr = {r_pre:.2f},  {p_str}  (n = {n_pre} bins)',
             fontweight='bold')
ax.legend(fontsize=10)

fig.savefig(OUT_DIR / 'preflood_station_comparison.png')
plt.show()
print(f'Pre-flood r = {r_pre:.3f},  {p_str},  n = {n_pre}')


## 14 · Pre vs. Post-flood Comparison per Station  
Mean counts before and after the flood at each station, plus a direct comparison during the post-flood overlapping window (Jun 12–15).  
**Caution:** different absolute dates — interpret as order-of-magnitude change only.

In [ ]:
from scipy.stats import mannwhitneyu

# ── Mean count per station × period (bar chart with Mann-Whitney p) ───────────
stats_rows = []
for tag, tl in TIMELINES.items():
    active = tl[tl['status'].isin(['normal', 'no_net', 'unknown'])]
    stats_rows.append({
        'tag':     tag,
        'station': DATASETS[tag]['station'],
        'period':  DATASETS[tag]['period'],
        'mean':    active['est_true'].mean(),
        'std':     active['est_true'].std(),
        'total':   int(active['est_true'].sum()),
        'n_bins':  len(active),
        'counts':  active['est_true'].values,
    })
stats_df = pd.DataFrame(stats_rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

for ax, (station_label, tag_pair) in zip(axes, [
    ('RFDO1  (Downstream)', ['250528_post', '250609_post']),
    ('RFUP1  (Upstream)',   ['250529_post', '250609_RFUP1']),
]):
    sub = stats_df[stats_df['tag'].isin(tag_pair)].copy()
    periods = sub['period'].tolist()
    means   = sub['mean'].tolist()
    stds    = sub['std'].tolist()
    colors  = [COLORS[t] for t in tag_pair]

    bars = ax.bar(periods, means, color=colors, alpha=0.82, width=0.45,
                  edgecolor='white', linewidth=0.5)
    ax.errorbar(periods, means, yerr=stds,
                fmt='none', color='#333333', capsize=6, lw=1.8)

    # Mann-Whitney U test: pre vs post count distributions
    pre_counts  = sub[sub['period'] == 'Pre-flood' ]['counts'].values[0]
    post_counts = sub[sub['period'] == 'Post-flood']['counts'].values[0]
    u_stat, p_mw = mannwhitneyu(pre_counts, post_counts, alternative='two-sided')
    p_mw_str = 'p < 0.001' if p_mw < 0.001 else f'p = {p_mw:.3f}'

    # Significance bracket
    y_top = max(means[0] + stds[0], means[1] + stds[1]) * 1.12
    ax.plot([0, 0, 1, 1], [y_top*0.92, y_top, y_top, y_top*0.92],
            color='black', lw=1.2)
    ax.text(0.5, y_top * 1.01, f'Mann-Whitney  {p_mw_str}',
            ha='center', va='bottom', fontsize=9.5)

    for bar, row in zip(bars, sub.itertuples()):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 0.02,
                f'n = {row.n_bins}', ha='center', va='bottom',
                fontsize=9, color='white', fontweight='bold')

    ax.set_ylabel('Mean est. insects / 30 min  (± SD)')
    ax.set_title(f'{station_label}\nPre- vs. Post-flood', fontweight='bold')
    ax.set_ylim(bottom=0)

fig.savefig(OUT_DIR / 'prepost_counts_per_station.png')
plt.show()

# ── Post-flood overlapping window (Jun 12–15) ─────────────────────────────────
post_ds = TIMELINES['250609_post'].copy()
post_us = TIMELINES['250609_RFUP1'].copy()

pds = post_ds[post_ds['status'].isin(['normal', 'no_net'])].set_index('bin_start')
pus = post_us[post_us['status'].isin(['normal', 'unknown'])].set_index('bin_start')

pov_start = max(pds.index.min(), pus.index.min())
pov_end   = min(pds.index.max(), pus.index.max())

ds_pov = pds.loc[pov_start:pov_end, 'est_true']
us_pov = pus.loc[pov_start:pov_end, 'est_true']

paired_post = pd.DataFrame({'DS': ds_pov, 'US': us_pov}).dropna()
n_post = len(paired_post)

if n_post > 2:
    r_post = np.corrcoef(paired_post['DS'], paired_post['US'])[0, 1]
    t2 = r_post * np.sqrt(n_post - 2) / np.sqrt(max(1 - r_post**2, 1e-12))
    from scipy.stats import t as _t
    p_post = 2 * _t.sf(abs(t2), df=n_post - 2)
    p_post_str = 'p < 0.001' if p_post < 0.001 else f'p = {p_post:.3f}'
    r_txt = f'r = {r_post:.2f},  {p_post_str}'
else:
    r_txt = 'n/a (too few bins)'

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

ax = axes[0]
ax.plot(ds_pov.index, ds_pov.values, color=COLORS['250609_post'], lw=2,
        label='RFDO1  Downstream')
ax.plot(us_pov.index, us_pov.values, color=COLORS['250609_RFUP1'], lw=2,
        label='RFUP1  Upstream', alpha=0.85)
_apply_date_xaxis(ax)
ax.set_ylabel('Est. insects / 30 min')
ax.set_title(f'Post-flood — both stations\n'
             f'{pov_start.strftime("%d %b")} – {pov_end.strftime("%d %b %Y")}',
             fontweight='bold')
ax.legend()

ax = axes[1]
if not paired_post.empty:
    ax.scatter(paired_post['DS'], paired_post['US'],
               alpha=0.45, color='tomato', s=30, edgecolors='none')
    lim2 = max(paired_post.max().max() * 1.1, 1)
    ax.plot([0, lim2], [0, lim2], 'k--', lw=1, alpha=0.4, label='1:1 line')
    ax.set_xlim(0, lim2); ax.set_ylim(0, lim2)
ax.set_xlabel('RFDO1  Downstream (est. insects / 30 min)')
ax.set_ylabel('RFUP1  Upstream (est. insects / 30 min)')
ax.set_title(f'Post-flood synchrony\n{r_txt}  (n = {n_post} bins)',
             fontweight='bold')
ax.legend(fontsize=10)

fig.savefig(OUT_DIR / 'postflood_station_comparison.png')
plt.show()
print(f'Post-flood: {pov_start.date()} – {pov_end.date()},  {n_post} paired bins')


## 15 · Diurnal Pattern — Pre vs. Post-flood per Station  
Solid line = pre-flood, dashed = post-flood. A phase shift (peak moves in time) would suggest a community composition change.  
**Caution:** ~5 days per curve — treat as indicative.

In [ ]:
# Diurnal curves: pre vs. post × station (4 lines, 2 panels)
# Only 'normal', 'no_net', 'unknown' bins — 'cleaning' bins are excluded.
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

for ax, (station_label, pairs) in zip(axes, [
    ('RFDO1  (Downstream)',
     [('Pre-flood',  '250528_post'),  ('Post-flood', '250609_post')]),
    ('RFUP1  (Upstream)',
     [('Pre-flood',  '250529_post'),  ('Post-flood', '250609_RFUP1')]),
]):
    peak_times = {}
    for period_label, tag in pairs:
        tl = TIMELINES[tag]
        active = tl[tl['status'].isin(['normal', 'unknown', 'no_net'])].copy()
        active['tod'] = active['bin_start'].dt.hour + active['bin_start'].dt.minute / 60
        diurnal = active.groupby('tod')['est_true'].agg(['mean', 'sem']).reset_index()
        peak_tod = diurnal.loc[diurnal['mean'].idxmax(), 'tod']
        peak_times[period_label] = peak_tod

        ls    = '-' if 'Pre' in period_label else '--'
        color = COLORS[tag]
        lbl   = f'{period_label}  (peak {int(peak_tod):02d}:{int((peak_tod%1)*60):02d})'
        ax.plot(diurnal['tod'], diurnal['mean'],
                lw=2.2, color=color, ls=ls, label=lbl)
        ax.fill_between(diurnal['tod'],
                        diurnal['mean'] - diurnal['sem'],
                        diurnal['mean'] + diurnal['sem'],
                        color=color, alpha=0.15)

    ax.set_xticks(range(0, 25, 3))
    ax.set_xticklabels([f'{h:02d}:00' for h in range(0, 25, 3)])
    ax.set_xlim(0, 24)
    ax.set_xlabel('Time of day')
    ax.set_ylabel('Mean est. insects / 30 min  (± SEM)')
    ax.set_title(f'{station_label}\nDiurnal pattern — pre vs. post flood\n'
                 f'Solid = pre-flood  |  Dashed = post-flood  |  ~5 days per curve',
                 fontweight='bold')
    ax.legend()

fig.savefig(OUT_DIR / 'diurnal_prepost_comparison.png')
plt.show()


## 16 · Total Count & Nocturnal Fraction

Overview of total insect passage per deployment and the proportion occurring at night (20:00–06:00). A nocturnal shift post-flood would suggest a change in dominant species.

In [ ]:
# Total estimated insects per dataset
fig, ax = plt.subplots(figsize=(10, 5), constrained_layout=True)

totals_rows = []
for tag, tl in TIMELINES.items():
    active = tl[tl['status'].isin(['normal', 'no_net', 'unknown'])]
    totals_rows.append({
        'label': (f"{DATASETS[tag]['station']}\n"
                  f"{DATASETS[tag]['period']}"),
        'total': int(active['est_true'].sum()),
        'color': COLORS[tag],
    })
tot_df = pd.DataFrame(totals_rows)

bars = ax.bar(tot_df['label'], tot_df['total'],
              color=tot_df['color'], alpha=0.85, width=0.5, edgecolor='white')
for bar, row in zip(bars, tot_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.01,
            f'{row.total:,}', ha='center', va='bottom', fontweight='bold')

ax.set_ylabel('Total estimated insects (full deployment)')
ax.set_title('Total insect count per deployment', fontweight='bold')
ax.set_ylim(bottom=0)

fig.savefig(OUT_DIR / 'total_counts_overview.png')
plt.show()


In [ ]:
# Nocturnal fraction (20:00–06:00) per dataset
from scipy.stats import chi2_contingency

noc_rows = []
for tag, tl in TIMELINES.items():
    active = tl[tl['status'].isin(['normal', 'no_net', 'unknown'])].copy()
    active['hour'] = active['bin_start'].dt.hour
    active['is_night'] = (active['hour'] >= 20) | (active['hour'] < 6)
    night = int(active.loc[active['is_night'],  'est_true'].sum())
    day   = int(active.loc[~active['is_night'], 'est_true'].sum())
    total = night + day
    noc_rows.append({
        'tag':    tag,
        'label':  f"{DATASETS[tag]['station']}\n{DATASETS[tag]['period']}",
        'station': DATASETS[tag]['station'],
        'period':  DATASETS[tag]['period'],
        'night':  night,
        'day':    day,
        'total':  total,
        'frac':   night / total if total > 0 else 0,
        'color':  COLORS[tag],
    })
noc_df = pd.DataFrame(noc_rows)

# Chi-square test: is the day/night split different pre vs post for each station?
chi_results = {}
for station, tags in [('RFDO1', ['250528_post', '250609_post']),
                      ('RFUP1', ['250529_post', '250609_RFUP1'])]:
    sub = noc_df[noc_df['tag'].isin(tags)]
    contingency = sub[['night', 'day']].values
    chi2, p, _, _ = chi2_contingency(contingency)
    chi_results[station] = p

fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)

# Left: nocturnal fraction bar
ax = axes[0]
bars = ax.bar(noc_df['label'], noc_df['frac'] * 100,
              color=noc_df['color'], alpha=0.85, width=0.5, edgecolor='white')
ax.axhline(50, color='black', lw=1.2, ls='--', alpha=0.45, label='50 %')
for bar, row in zip(bars, noc_df.itertuples()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
            f'{row.frac*100:.1f} %', ha='center', va='bottom')

# Chi-square brackets
pairs = [([0, 1], 'RFDO1'), ([2, 3], 'RFUP1')]
y_tops = [96, 96]
for (idxs, station), y_top in zip(pairs, y_tops):
    p_val = chi_results[station]
    p_str = 'p < 0.001' if p_val < 0.001 else f'p = {p_val:.3f}'
    x0, x1 = idxs
    ax.plot([x0, x0, x1, x1], [y_top-4, y_top, y_top, y_top-4], color='black', lw=1.2)
    ax.text((x0 + x1) / 2, y_top + 0.5, f'χ²  {p_str}', ha='center', va='bottom', fontsize=10)

ax.set_ylabel('Nocturnal fraction (%)  —  20:00–06:00')
ax.set_title('Proportion of insects passing at night', fontweight='bold')
ax.set_ylim(0, 108)
ax.legend()

# Right: stacked day / night totals
ax = axes[1]
x = np.arange(len(noc_df))
ax.bar(x, noc_df['night'], width=0.5, label='Night  (20:00–06:00)',
       color='#2c3e50', alpha=0.85)
ax.bar(x, noc_df['day'],   width=0.5, bottom=noc_df['night'],
       label='Day  (06:00–20:00)', color='#f39c12', alpha=0.75)
ax.set_xticks(x)
ax.set_xticklabels(noc_df['label'])
ax.set_ylabel('Total estimated insects')
ax.set_title('Day vs. night totals per deployment', fontweight='bold')
ax.legend()

fig.savefig(OUT_DIR / 'nocturnal_fraction.png')
plt.show()

print('\nNocturnal fractions:')
for row in noc_df.itertuples():
    print(f'  {row.tag}: {row.frac*100:.1f} % nocturnal  ({row.night:,} night / {row.day:,} day)')
print('\nChi-square (day/night split pre vs post):')
for station, p in chi_results.items():
    p_str = '< 0.001' if p < 0.001 else f'{p:.3f}'
    print(f'  {station}: p = {p_str}')
